In [2]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

X = pd.read_csv('train.csv')
cat_features = ['Sex', 'Chest pain type', 'FBS over 120', 'EKG results', 'Exercise angina', 'Slope of ST',
                'Number of vessels fluro', 'Thallium', 'Age_bin', 'BP_bin', 'Cholesterol_bin', 'MaxHR_bin',
                'ST_bin']

#Bin
X['Age_bin'] = pd.cut(X['Age'], bins=5).astype(str)
X['BP_bin'] = pd.cut(X['BP'], bins=5).astype(str)
X['Cholesterol_bin'] = pd.cut(X['Cholesterol'], bins=5).astype(str)
X['MaxHR_bin'] = pd.cut(X['Max HR'], bins=5).astype(str)
X['ST_bin'] = pd.cut(X['ST depression'], bins=5).astype(str)

#Digit
X["Age_units"] = X["Age"] % 10
X["Age_tens"] = (X["Age"] // 10) % 10

X["BP_units"] = X["BP"] % 10
X["BP_tens"] = (X["BP"] // 10) % 10
X["BP_hundreds"] = (X["BP"] // 100) % 10

X["Chol_units"] = X["Cholesterol"] % 10
X["Chol_tens"] = (X["Cholesterol"] // 10) % 10
X["Chol_hundreds"] = (X["Cholesterol"] // 100) % 10

X["MaxHR_units"] = X["Max HR"] % 10
X["MaxHR_tens"] = (X["Max HR"] // 10) % 10
X["MaxHR_hundreds"] = (X["Max HR"] // 100) % 10

X["St_depression_1"] = (X["ST depression"] * 10 % 10).astype(int)

X.replace({'Heart Disease': {'Presence': 1, 'Absence': 0}}, inplace=True)
y = X['Heart Disease'].astype(int)
X.drop(['Heart Disease', 'id'], axis=1, inplace=True)

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model_scores = []

for train_index, test_index in kf.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    train_pool = Pool(X_train, y_train, cat_features=cat_features)
    test_pool = Pool(X_test, y_test, cat_features=cat_features)

    model = CatBoostClassifier(iterations=1000,
                               task_type="GPU",
                               devices='0',
                               loss_function='Logloss',
                               verbose=False,
                               random_seed=42)

    model.fit(train_pool)

    preds = model.predict_proba(test_pool)[:, 1]
    score = roc_auc_score(y_test, preds)
    model_scores.append(score)

print(f'Средняя точность: {np.mean(model_scores)}')
model.save_model('heart_disease_model.cbm')

Средняя точность: 0.9554221631718409


In [21]:
X = pd.read_csv('test.csv')

#Bin
X['Age_bin'] = pd.cut(X['Age'], bins=5).astype(str)
X['BP_bin'] = pd.cut(X['BP'], bins=5).astype(str)
X['Cholesterol_bin'] = pd.cut(X['Cholesterol'], bins=5).astype(str)
X['MaxHR_bin'] = pd.cut(X['Max HR'], bins=5).astype(str)
X['ST_bin'] = pd.cut(X['ST depression'], bins=5).astype(str)

#Digit
X["Age_units"] = X["Age"] % 10
X["Age_tens"] = (X["Age"] // 10) % 10

X["BP_units"] = X["BP"] % 10
X["BP_tens"] = (X["BP"] // 10) % 10
X["BP_hundreds"] = (X["BP"] // 100) % 10

X["Chol_units"] = X["Cholesterol"] % 10
X["Chol_tens"] = (X["Cholesterol"] // 10) % 10
X["Chol_hundreds"] = (X["Cholesterol"] // 100) % 10

X["MaxHR_units"] = X["Max HR"] % 10
X["MaxHR_tens"] = (X["Max HR"] // 10) % 10
X["MaxHR_hundreds"] = (X["Max HR"] // 100) % 10


id = X['id']
X.drop('id', axis=1, inplace=True)
preds = model.predict_proba(X)[:, 1]

sample_submission = pd.DataFrame(data={'id': id, 'Heart Disease': preds})
sample_submission.to_csv('sample_submission.csv', index=False)